# 1 · Expression matrix and sample annotation

**Matrix.** Ensembl identifiers are mapped to symbols through GENCODE v36, only
protein-coding genes are kept (18,573), and values are transformed to log2(TPM + 1).
No batch correction or imputation is applied: the analysis measures how much
variance the technical variables carry.

**Cohort.** Of the 915 delivered samples, the 14 from two non-public sequencing
centres are excluded, leaving 901: 654 tumours, 118 normal tissues and 129 cell lines.

**Annotation.** The run sheet supplies sample type, diagnosis, library chemistry,
RIN, sequencing kit and flowcell; a blank Sample Type denotes normal tissue. Two
fields are derived here: the per-sample fraction of genes at zero TPM (sparsity),
and sex, inferred from the mean expression of seven Y-linked genes
(Supplementary Methods §2).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # make `analysis` importable from notebooks/

from analysis import notebook as nb
nb.setup()

working in sarcoma/


In [2]:
from analysis import build

expression, meta = build.build_matrix()
print(expression.shape)
meta["sample_type"].value_counts()

1. gene annotation (GENCODE v36, local)


   60616 genes, 19944 protein-coding
2. TPM matrix


   33967 genes x 915 samples
   unmapped ensembl ids: 0
   protein-coding, symbol-indexed: 18573 genes


3. run sheet, restricted to the public cohort
   dropping 14 sample(s) from non-public centres: {'UCalgary_CHGI': 13, 'UdeMontreal_IRIC': 1}
   901 samples from RNAseq_Landscape_Manuscript
4. per-sample sparsity
   zero fraction: median 0.124, max 0.307
5. log2(TPM+1)


6. metadata
   flowcell: 780 samples, 92 runs (85 SRA-accession values dropped, 118 recovered from the sample id)
   sex inferred from 7 Y-linked genes: 492 male, 400 female, 9 between the thresholds and left unassigned
   blank Sample Type -> NORMAL: 118 sample(s)


   901 samples x 16 annotation columns
7. summaries


sample_type  CELL_LINE  NORMAL  TUMOR  total
diagnosis                                   
NBL                 39       0    206    245
RMS                 32       0     97    129
unlabelled           0     118      4    122
ES                  43       0     77    120
OS                   9       0     93    102
DSRCT                0       0     40     40
SYNS                 3       0     29     32
LIHB                 0       0     30     30
WT                   0       0     27     27
ASPS                 3       0     16     19
CCSK                 0       0     13     13
MEL                  0       0      6      6
SARCNOS              0       0      6      6
YST                  0       0      5      5
CCS                  0       0      3      3
ELS                  0       0      2      2

   library type x sparsity


                count  median
library_type                 
polya             571   0.113
polya_stranded      1   0.126
ribominus          36   0.200
ribozero          175   0.181

   matched contrast groups
contrast_group
other           407
NBL_Tumor       206
RMS_Tumor        97
ES_Tumor         77
ES_CellLine      43
NBL_CellLine     39
RMS_CellLine     32

  wrote Z:\postdoc\projects\PathwayTheme\paper\sarcoma\data\processed\expression_log2.tsv and metadata.tsv
(18573, 901)


sample_type
TUMOR        654
CELL_LINE    129
NORMAL       118
Name: count, dtype: int64

**External annotation.** Brohl *et al.* Table S1 and the open TARGET clinical tables
are joined on the sample and patient identifiers. They supply race, ethnicity,
vital status, age, disease at diagnosis, the normals' tissue of origin and 15
alignment metrics, complete library chemistry, RIN and sequencing kit where the run
sheet is blank, and provide the 85 recorded sexes against which the inferred sex is
checked (Supplementary Methods §9).

In [3]:
from analysis import link

annotation = link.link_supplement()
nb.show_table(annotation, name="clinical_annotation.tsv")

-> Table S1A_Patient Samples: 935 rows x 6 cols
   via SampleID                 matched  805/901


   via Patient ID               matched   96/901


-> Table S1B.RNA-Seq_QC: 935 rows x 24 cols


   via SampleID                 matched  805/901
   via Sample.Data.ID           matched    0/901


   via Patient.ID               matched   96/901
-> Table S1D. OS_Clinical: 92 rows x 29 cols
   via TARGET USI/Patient ID    matched   85/901


-> GDC TARGET clinical (2249 USIs across 5 projects)
   matched 246/901 on run-sheet patient_id = USI; filled 919 blanks, 311 agreed with Brohl, 0 conflicted
   sex: recorded for 85, the inferred call agrees on 85/85 of them; +807 inferred, 892 of 901 now assigned
-> completing the run sheet
   library_type 783 -> 901 (+118 from Brohl S1A); 40/783 disagree where both record one, run sheet kept
   rin 355 -> 631 (+276 from Brohl S1B); identical on 322/334 of the overlap
   seq_kit 494 -> 596 (+102 propagated along 65 single-kit flowcells, 0 conflicting)


   rewrote Z:\postdoc\projects\PathwayTheme\paper\sarcoma\data\processed\metadata.tsv with the completed fields



coverage over our 901 samples:
                  field  n_annotated  pct_of_cohort  n_levels
   brohl_tissue_subtype          901          100.0        23
        brohl_diagnosis          901          100.0        15
   brohl_diagnosis_code          901          100.0        18
     brohl_library_type          901          100.0         3
      brohl_tissue_type          901          100.0         3
              brohl_rin          610           67.7        50
        qc_mapping_rate          901          100.0       873
         qc_unique_rate          901          100.0       875
    qc_duplication_rate          901          100.0       875
         qc_exonic_rate          901          100.0       875
       qc_intronic_rate          901          100.0       875
     qc_intragenic_rate          901          100.0       875
     qc_intergenic_rate          901          100.0       875
qc_fragment_length_mean          901          100.0       225
         qc_read_length          901  

**clinical_annotation.tsv** — 901 rows × 33 columns

,brohl_tissue_subtype,brohl_diagnosis,brohl_diagnosis_code,brohl_library_type,brohl_tissue_type,brohl_rin,qc_mapping_rate,qc_unique_rate,qc_duplication_rate,qc_exonic_rate,qc_intronic_rate,qc_intragenic_rate,qc_intergenic_rate,qc_fragment_length_mean,qc_read_length,...,qc_base_mismatch_rate,qc_mean_per_base_cov,qc_rrna,sex,race,ethnicity,disease_at_diagnosis,age_at_diagnosis_days,vital_status,gdc_primary_diagnosis,gdc_project,age_at_diagnosis_years,tissue_of_origin,sex_recorded,sex_source
sample_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
6647seq_T_D2AHCACXX,Cell Line,Ewing Sarcoma,EWS,Ribozero Stranded,CellLine,NaN,0.945013,0.649761,0.350239,0.528316,0.190291,0.718607,0.280792,271,101,...,0.001811,122.670270,3172339,Female,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,Y-linked expression
7556UK_T_C628WANXX,Cell Line,Rhabdomyosarcoma,RMS.FN,Ribozero Stranded,CellLine,10.0,0.894263,0.613192,0.386808,0.604678,0.124553,0.729232,0.269463,141,125,...,0.001946,56.144867,1743500,Female,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,Y-linked expression
A673seq_T_D26GWACXX,Cell Line,Ewing Sarcoma,EWS,Ribozero Stranded,CellLine,NaN,0.706796,0.740264,0.259736,0.529465,0.273260,0.802725,0.196770,248,101,...,0.001712,101.282690,2118273,Female,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,Y-linked expression
ASPS001tumor_T_D1PR6ACXX,Tumor,Alveolar soft part sarcoma,ASPS,PolyA,Tumor,9.0,0.971427,0.368519,0.631481,0.682819,0.032597,0.715416,0.284082,294,101,...,0.002452,125.239690,11590158,Female,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,Y-linked expression
ASPS002tumor_T_D1PR6ACXX,Tumor,Alveolar soft part sarcoma,ASPS,PolyA,Tumor,9.5,0.970030,0.631791,0.368209,0.694557,0.068466,0.763023,0.236447,298,101,...,0.002468,147.599610,5042386,Female,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,Y-linked expression
ASPS003tumor_T_D1UALACXX,Tumor,Alveolar soft part sarcoma,ASPS,PolyA,Tumor,NaN,0.954869,0.387629,0.612371,0.396624,0.063162,0.459787,0.539934,283,101,...,0.003110,92.927414,12418056,Male,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,Y-linked expression
ASPS004tumor_T_D1UALACXX,Tumor,Alveolar soft part sarcoma,ASPS,PolyA,Tumor,NaN,0.959497,0.721960,0.278040,0.789821,0.072921,0.862742,0.136604,290,101,...,0.002730,138.097090,3081204,Male,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,Y-linked expression
ASPS005tumor_T_D1UALACXX,Cell Line,Alveolar soft part sarcoma,ASPS,Ribozero Stranded,CellLine,NaN,0.922631,0.551125,0.448875,0.460325,0.160590,0.620916,0.378329,265,101,...,0.002031,114.907160,14883845,Female,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,Y-linked expression
ASPS007tumor_T_D1UALACXX,Tumor,Alveolar soft part sarcoma,ASPS,PolyA,Tumor,NaN,0.950793,0.733004,0.266996,0.728981,0.082295,0.811276,0.188218,240,101,...,0.003527,86.896820,2428817,Male,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,Y-linked expression
